In [ ]:
"""
Requirements:
    pip install numpy pandas scikit-learn
"""

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------------------------------------------------------
# Sample Dataset
# ---------------------------------------------------------------------------

MOVIES = pd.DataFrame({
    "movie_id": range(1, 11),
    "title": [
        "The Matrix", "Inception", "Interstellar", "The Dark Knight",
        "Avengers: Endgame", "Titanic", "The Notebook", "La La Land",
        "Toy Story", "Finding Nemo"
    ],
    "genres": [
        "sci-fi action thriller",
        "sci-fi thriller mystery",
        "sci-fi drama adventure",
        "action thriller crime",
        "action sci-fi superhero",
        "romance drama",
        "romance drama",
        "romance musical drama",
        "animation adventure comedy family",
        "animation adventure comedy family"
    ]
})

# User-Movie Ratings (0 = not rated)
RATINGS = pd.DataFrame({
    "user":      [1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5],
    "movie_id":  [1, 2, 3, 4, 1, 3, 5, 2, 4, 6, 7, 5, 6, 8, 9, 10, 7],
    "rating":    [5, 4, 5, 3, 4, 5, 3, 5, 4, 2, 3, 5, 4, 5, 5,  4, 3]
})

# ---------------------------------------------------------------------------
# Collaborative Filtering
# ---------------------------------------------------------------------------

def build_user_movie_matrix():
    matrix = RATINGS.pivot_table(
        index="user", columns="movie_id", values="rating", fill_value=0
    )
    return matrix

def collaborative_recommendations(user_id: int, top_n: int = 3):
    matrix = build_user_movie_matrix()

    if user_id not in matrix.index:
        return []

    user_sim = cosine_similarity(matrix)
    sim_df = pd.DataFrame(user_sim, index=matrix.index, columns=matrix.index)

    # Weighted average of other users' ratings
    sim_scores = sim_df[user_id].drop(user_id)
    weighted = matrix.drop(user_id).T.dot(sim_scores)
    seen = matrix.loc[user_id]
    unseen = weighted[seen == 0]
    top_ids = unseen.nlargest(top_n).index.tolist()
    return MOVIES[MOVIES["movie_id"].isin(top_ids)]["title"].tolist()

# ---------------------------------------------------------------------------
# Content-Based Filtering
# ---------------------------------------------------------------------------

def build_tfidf_matrix():
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(MOVIES["genres"])
    return tfidf_matrix

def content_based_recommendations(movie_title: str, top_n: int = 3):
    matches = MOVIES[MOVIES["title"].str.lower() == movie_title.lower()]
    if matches.empty:
        return []

    idx = matches.index[0]
    tfidf_matrix = build_tfidf_matrix()
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores[idx] = 0  # Exclude self
    top_indices = sim_scores.argsort()[-top_n:][::-1]
    return MOVIES.iloc[top_indices]["title"].tolist()

# ---------------------------------------------------------------------------
# Interactive CLI
# ---------------------------------------------------------------------------

def print_movies():
    print("\nAvailable Movies:")
    print("-" * 40)
    for _, row in MOVIES.iterrows():
        print(f"  [{row['movie_id']:2d}] {row['title']}")
    print()

def main():
    print("=" * 50)
    print("    CodSoft Movie Recommendation System")
    print("=" * 50)

    while True:
        print("\nOptions:")
        print("  1 → Collaborative Filtering (by user)")
        print("  2 → Content-Based Filtering (by movie)")
        print("  3 → Show all movies")
        print("  4 → Exit")

        choice = input("\nYour choice: ").strip()

        if choice == "1":
            print("\nAvailable users: 1, 2, 3, 4, 5")
            try:
                uid = int(input("Enter user ID: ").strip())
                recs = collaborative_recommendations(uid)
                if recs:
                    print(f"\n🎬 Recommendations for User {uid}:")
                    for r in recs:
                        print(f"   • {r}")
                else:
                    print("No recommendations found (user not found or already seen all).")
            except ValueError:
                print("Please enter a valid user ID.")

        elif choice == "2":
            print_movies()
            title = input("Enter movie title: ").strip()
            recs = content_based_recommendations(title)
            if recs:
                print(f"\n🎬 Movies similar to '{title}':")
                for r in recs:
                    print(f"   • {r}")
            else:
                print("Movie not found. Check the title and try again.")

        elif choice == "3":
            print_movies()

        elif choice == "4":
            print("Goodbye! 👋")
            break

        else:
            print("Invalid choice. Please enter 1, 2, 3, or 4.")

if __name__ == "__main__":
    main()